# O revisor do Nilo — treinar e medir

Treina um revisor pequeno e dedicado para o Andar 10 de *The Normal Elevator*, e
mede o resultado **com a mesma régua do jogo**.

**O que este caderno mede e o que ele NÃO mede.** Ele mede QUALIDADE: quantos
defeitos o modelo conserta, quantos ecos, cópias, promessas e quebras de cânone
ele comete. Ele **não** mede velocidade — TURNO só existe no navegador com o
wllama, e GPU de Colab não diz nada sobre o celular de ninguém. A carga continua
prevista pelo tamanho do arquivo (MB ÷ 32 ≈ segundos).

**Por que o julgamento roda em `node` e não em Python.** A régua mora em
`bancada-navegador/defeitos.mjs`, ao lado do cânone do jogo. Portar para Python
criaria uma segunda cópia para divergir — este projeto já pagou esse preço duas
vezes, quando a cópia da bancada ficou mais frouxa que o cânone do jogo e o
placar mentiu em dois modelos. Uma régua, um arquivo.

GPU: T4 (grátis) treina o 360M em poucos minutos; L4 usa bf16 e é mais rápida.


In [ ]:
!nvidia-smi -L || echo "SEM GPU — vá em Ambiente de execução › Alterar tipo › GPU"
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Dependências

`sentencepiece` não é usado pelo tokenizador do SmolLM2, mas o conversor do
llama.cpp tenta o caminho sentencepiece ANTES do BPE e só cai para o certo se o
import funcionar — sem ele a conversão morre com `ModuleNotFoundError`.

In [ ]:
!pip -q install "transformers>=4.44" peft accelerate sentencepiece gguf
# O julgamento roda em node. O Colab costuma já ter; se não tiver, instala.
!node --version || (apt-get -qq update && apt-get -qq install -y nodejs)

## 2. O repositório

O corpus, a prova e a régua vivem na branch de trabalho. Se o repositório for
privado, gere um token com escopo `repo` e cole em `TOKEN`.

In [ ]:
TOKEN = ""   # deixe vazio se o repositório for público
REPO  = "Felipe9272727/Jdjdjddj"
BRANCH = "claude/persistent-download-storage-i6l88v"

url = f"https://{TOKEN + '@' if TOKEN else ''}github.com/{REPO}.git"
!rm -rf jdjdjddj && git clone -q --depth 1 -b {BRANCH} {url} jdjdjddj
%cd jdjdjddj/jubileu/bancada-navegador
!ls corpus

## 3. O corpus, conferido antes de treinar

`conferir.mjs` falha alto em dois casos, e os dois arruinam o treino em silêncio:

1. **treinar na prova** — se um caso da prova vazar para o corpus, o placar
   depois vira enfeite;
2. **treinar o defeito** — se uma frase "certa" quebrar o cânone, o modelo
   aprende a quebrá-lo com convicção.

In [ ]:
!node corpus/conferir.mjs && node corpus/gerar.mjs
!head -c 600 corpus/treino.jsonl

## 4. Treinar

LoRA, e não afinação inteira: com poucas centenas de linhas, mexer em todos os
pesos apaga o inglês que a gente veio buscar. A perda só conta na RESPOSTA — sem
a máscara o modelo gasta capacidade aprendendo a prever o enunciado, que ele
nunca vai precisar escrever.

Bases que já converteram para gguf sem susto (arquitetura `llama`, o caminho
mais batido): `HuggingFaceTB/SmolLM2-360M-Instruct` (386 MB em q8 ≈ 12 s de
carga) e `HuggingFaceTB/SmolLM2-135M-Instruct` (≈ 145 MB ≈ 5 s).

In [ ]:
import os
os.environ["MODELO"]  = "HuggingFaceTB/SmolLM2-360M-Instruct"
os.environ["EPOCAS"]  = "8"
os.environ["LOTE"]    = "8"     # numa T4 cabe folgado; na CPU use 4
os.environ["ACUMULA"] = "2"
os.environ["LR"]      = "2e-4"
os.environ["SAIDA"]   = "corpus/revisor-360m"
!python3 corpus/treinar.py

## 5. A prova, e o julgamento com a régua do jogo

24 casos (os 6 históricos + 18 novos) e 3 controles — frases que já estavam
certas, onde a única falha possível é ESTRAGAR.

**Leia as saídas reprovadas.** A régua pega palavra proibida, eco, cópia,
fragmento e promessa. Ela não pega *"a few steps from a door that does not
exist"* — que é canônicamente errado (a porta existe, ela só não abre) e passa
liso. Modelo treinado erra em frase plausível, e é aí que a régua enxerga pior.

In [ ]:
!MODELO=corpus/revisor-360m SAIDAS=corpus/saidas.jsonl python3 corpus/gerar-saidas.py
!node corpus/julgar-saidas.mjs corpus/saidas.jsonl

## 6. Para gguf, que é o que o jogo carrega

O conversor do llama.cpp deixou de ser um arquivo só: hoje `convert_hf_to_gguf.py`
importa o pacote `conversion/`, então baixar o script avulso não converte nada.
`--outtype q8_0` quantiza na conversão e evita compilar o `llama-quantize`.

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!LLAMACPP=/content/llama.cpp bash corpus/para-gguf.sh corpus/revisor-360m revisor.gguf
from google.colab import files; files.download("revisor.gguf")

## 7. O que fazer com o arquivo

Suba o `revisor.gguf` para um repositório de modelos e aponte a entrada do
revisor em `src/npc/floor10Brains.ts` para a URL. A medição de TURNO (carga +
1ª chamada fria) é feita na bancada do navegador, não aqui:

```
PROVA=grande RODADAS=2 ENUNCIADO=treinado SISTEMA=treinado TEMPERATURA=0 \
  MODELOS="revisor.gguf:REVISOR:q8_0:1024" node revisor-candidatos.mjs
```
